In [0]:
%pip install azure-eventhub requests sseclient-py

In [0]:
import json
import requests
from sseclient import SSEClient
from azure.eventhub import EventHubProducerClient, EventData

In [0]:
dbutils.widgets.text("secret_scope", "scope-trezio2005")
secret_scope = dbutils.widgets.get("secret_scope")

dbutils.widgets.text("evh_name", "trezio2005_evh")
evh_name = dbutils.widgets.get("evh_name")

conn_string = dbutils.secrets.get(scope=secret_scope, key="trezio2005-evh-cs")

In [0]:
def run_producer(max_events=100):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=conn_string,
        eventhub_name=evh_name
    )

    url = "https://stream.wikimedia.org/v2/stream/recentchange"
    headers = {
        "User-Agent": "Databricks-Student-App (trezio2005@gmail.com)"
    }
    response = requests.get(url, stream=True, headers=headers)
    client = SSEClient(response)
    
    events_sent=0
    with producer:
        for event in client.events():
            if event.event == "message":
                try:
                    change_data = json.loads(event.data)

                    simplified_event = {
                        "user": change_data.get("user"),
                        "title": change_data.get("title"),
                        "type": change_data.get("type"),
                        "server_url": change_data.get("server_url"),
                        "timestamp": change_data.get("meta", {}).get("dt")
                    }

                    event_data_batch = producer.create_batch()
                    event_data_batch.add(EventData(json.dumps(simplified_event)))

                    producer.send_batch(event_data_batch)
                    events_sent += 1

                    if events_sent >= max_events:
                        print(f"Sent {events_sent} events.")
                        break

                except ValueError:
                    pass

run_producer(100)